# Interactuar con un LLM de Databricks mediante llamadas a la API usando el SDK de Databricks

En este notebook aprenderás a **consultar modelos de lenguaje (LLMs)** servidos en Databricks de tres formas:

1. **Con el SDK de Python de Databricks** (`databricks-sdk`): la forma programática y recomendada desde notebooks o aplicaciones Python.
2. **Con SQL** mediante la función `ai_query()`: ideal para analistas y pipelines de datos.
3. (Como referencia) con la **API REST** compatible con OpenAI que expone Databricks Model Serving.

## ¿Para qué sirve esto?

Databricks **Model Serving** (Mosaic AI Model Serving) expone los modelos a través de *serving endpoints*. Esto incluye:

- **Foundation Model APIs**: modelos prealojados y listos para usar (Claude, Llama, GPT-OSS, etc.) facturados por tokens consumidos (*pay-per-token*).
- **Provisioned Throughput**: capacidad reservada con rendimiento garantizado para cargas de producción.
- **External Models**: endpoints que actúan como proxy hacia proveedores externos (OpenAI, Anthropic, Google, etc.) gestionando credenciales de forma centralizada.

> 💡 La gran ventaja es que **no necesitas gestionar API keys manualmente**: el SDK usa la autenticación de tu workspace (notebook, token, OAuth o variables de entorno) de forma automática.

## 1. Instalación de dependencias y librerías

Instalamos el **SDK de Databricks** para Python. Fijamos una versión concreta (`==0.77.0`) para garantizar reproducibilidad y evitar que un cambio de API rompa el notebook.

- `%pip install` instala el paquete en el entorno del notebook actual (no a nivel de clúster).
- Para usar siempre la última versión podrías hacer `%pip install databricks-sdk --upgrade`, pero fijar la versión es una buena práctica en producción.

In [ ]:
# Instala el SDK oficial de Databricks para Python (versión fijada para reproducibilidad)
%pip install databricks-sdk==0.77.0

## 2. Reiniciar el entorno de Python

Tras instalar una librería con `%pip`, hay que **reiniciar el intérprete de Python** para que el nuevo paquete quede disponible y se eviten conflictos de versiones cargadas previamente.

- `dbutils.library.restartPython()` reinicia solo el proceso de Python del notebook (mantiene el clúster y la sesión de Spark).
- ⚠️ Importante: al reiniciar **se pierden todas las variables en memoria**. Por eso esta celda va siempre justo después del `%pip install` y antes de empezar a importar y crear objetos.

In [ ]:
# Reinicia el intérprete de Python para cargar la librería recién instalada
dbutils.library.restartPython()

## 3. Crear un Workspace Client

El `WorkspaceClient` es el **punto de entrada principal del SDK**: representa la conexión autenticada con tu workspace de Databricks y da acceso a todas las APIs (serving endpoints, clústeres, jobs, Unity Catalog, etc.).

### Autenticación automática
Dentro de un notebook de Databricks, `WorkspaceClient()` se autentica **automáticamente** con tus credenciales. Fuera de Databricks (por ejemplo, desde tu máquina local) usaría, por orden:
- Variables de entorno (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`).
- Perfiles del archivo `~/.databrickscfg`.
- OAuth (U2M / M2M con service principals).

### Sobre los imports
- `ChatMessage`: representa un mensaje del chat (contenido + rol).
- `ChatMessageRole`: enumera los roles posibles → `SYSTEM` (instrucciones de comportamiento), `USER` (lo que pregunta el usuario) y `ASSISTANT` (respuestas previas del modelo, útiles para dar contexto en conversaciones multivuelta).

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Crea el cliente autenticado con el workspace actual
w = WorkspaceClient()

# Opcional: listar los serving endpoints disponibles para ver qué modelos puedes consultar
for endpoint in w.serving_endpoints.list():
    print(endpoint.name)

## 4. Enviar una petición de Chat Completion al LLM

Usamos `w.serving_endpoints.query()` para enviar una conversación al modelo y obtener su respuesta.

### Parámetros principales de `query()`
| Parámetro | Descripción |
|-----------|-------------|
| `name` | Nombre del *serving endpoint* a consultar (p. ej. `databricks-claude-sonnet-4-5`). |
| `messages` | Lista de `ChatMessage` con la conversación (roles `system` / `user` / `assistant`). |
| `max_tokens` | Máximo de tokens a generar en la respuesta. |
| `temperature` | Aleatoriedad/creatividad (0 = determinista, ~1 = más creativo). |
| `top_p` | Muestreo por núcleo (alternativa a `temperature`). |
| `stop` | Lista de secuencias que detienen la generación. |
| `stream` | Si es `True`, devuelve la respuesta en *streaming* (token a token). |
| `n` | Número de respuestas alternativas a generar. |

### Estructura de la respuesta
La respuesta sigue el formato compatible con OpenAI:
- `response.choices[0].message.content` → el texto generado.
- `response.usage` → tokens de entrada/salida consumidos (útil para controlar costes).

> 💡 El mensaje `SYSTEM` define el "rol" o personalidad del modelo. Aquí le pedimos que actúe como Batman.

In [ ]:
response = w.serving_endpoints.query(
    name="databricks-claude-sonnet-4-5",
    messages=[
        # Mensaje SYSTEM: define el comportamiento/personalidad del modelo
        ChatMessage(role=ChatMessageRole.SYSTEM, content="Eres Batman, el protector de Gotham City. Responde en español."),
        # Mensaje USER: la pregunta del usuario
        ChatMessage(role=ChatMessageRole.USER, content="¿Cómo va hoy Gotham City?")
    ],
    max_tokens=1024,
    temperature=0.7,   # Controla la creatividad de la respuesta
)

print(f"RESPUESTA:\n{response.choices[0].message.content}")

# Información de consumo de tokens (útil para estimar costes)
if response.usage:
    print(f"\nTokens de entrada: {response.usage.prompt_tokens}")
    print(f"Tokens de salida: {response.usage.completion_tokens}")
    print(f"Tokens totales: {response.usage.total_tokens}")

## 5. Consultar el LLM usando SQL con `ai_query()`

Databricks permite invocar modelos **directamente desde SQL** con la función `ai_query()`. Esto es muy potente porque puedes aplicar un LLM **sobre columnas de una tabla** dentro de un pipeline de datos, sin escribir Python.

### Sintaxis
```sql
ai_query(endpoint, request [, returnType, failOnError, modelParameters])
```

| Argumento | Descripción |
|-----------|-------------|
| `endpoint` | Nombre del *serving endpoint* (string). |
| `request` | El prompt o la columna de texto a enviar al modelo. |
| `returnType` | (Opcional) Tipo de dato esperado de la respuesta, p. ej. `'STRING'` o un esquema estructurado. |
| `failOnError` | (Opcional) Si `false`, los errores por fila no detienen toda la consulta. |
| `modelParameters` | (Opcional) `struct` con `max_tokens`, `temperature`, etc. |

### Casos de uso típicos
- **Clasificación / análisis de sentimiento** de reseñas: `ai_query('endpoint', 'Clasifica el sentimiento: ' || texto_reseña)`.
- **Resumen** de documentos en lote.
- **Extracción de entidades** o traducción de columnas completas.

> 💡 También existen funciones de IA especializadas más sencillas: `ai_analyze_sentiment()`, `ai_classify()`, `ai_summarize()`, `ai_translate()`, `ai_extract()`, `ai_gen()` y `ai_similarity()`.

In [ ]:
%sql
-- Consulta simple: enviamos un prompt al modelo y recibimos texto
SELECT ai_query(
    "databricks-claude-sonnet-4-5",
    "Cuéntame algo interesante sobre Gotham City. Responde en español."
  ) AS respuesta;

## 6. (Extra) Conversación multivuelta

Para mantener el contexto de una conversación, vamos acumulando los mensajes (incluyendo las respuestas previas del `ASSISTANT`). El modelo en sí **no tiene memoria**: hay que reenviarle todo el historial en cada llamada.

In [ ]:
# Mantenemos un historial de mensajes que crece con cada turno
historial = [
    ChatMessage(role=ChatMessageRole.SYSTEM, content="Eres Batman. Responde en español, breve y en tono serio."),
    ChatMessage(role=ChatMessageRole.USER, content="¿Quién es tu mayor enemigo?"),
]

# Primer turno
r1 = w.serving_endpoints.query(name="databricks-claude-sonnet-4-5", messages=historial, max_tokens=256)
respuesta1 = r1.choices[0].message.content
print(f"Batman: {respuesta1}\n")

# Añadimos la respuesta del asistente y una nueva pregunta del usuario
historial.append(ChatMessage(role=ChatMessageRole.ASSISTANT, content=respuesta1))
historial.append(ChatMessage(role=ChatMessageRole.USER, content="¿Y cómo lo detienes habitualmente?"))

# Segundo turno: el modelo ya conoce el contexto anterior
r2 = w.serving_endpoints.query(name="databricks-claude-sonnet-4-5", messages=historial, max_tokens=256)
print(f"Batman: {r2.choices[0].message.content}")

## 7. (Extra) `ai_query()` con parámetros del modelo y sobre una tabla

`ai_query()` admite ajustar parámetros del modelo y procesar **muchas filas a la vez**. En este ejemplo creamos datos de prueba y pedimos al modelo que clasifique el sentimiento de cada reseña.

In [ ]:
%sql
-- Aplicamos el LLM fila a fila sobre una tabla de reseñas en memoria.
-- Pasamos modelParameters para limitar la longitud y bajar la temperatura.
WITH reseñas AS (
  SELECT * FROM VALUES
    ('El servicio fue excelente y muy rápido'),
    ('Una experiencia horrible, no vuelvo más'),
    ('El producto está bien, nada del otro mundo')
  AS t(texto)
)
SELECT
  texto,
  ai_query(
    'databricks-claude-sonnet-4-5',
    'Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO. Responde solo con una palabra. Reseña: ' || texto,
    modelParameters => named_struct('max_tokens', 5, 'temperature', 0.0)
  ) AS sentimiento
FROM reseñas;

## Resumen y buenas prácticas

| Enfoque | Cuándo usarlo |
|---------|---------------|
| **SDK Python** (`w.serving_endpoints.query`) | Aplicaciones, agentes, control fino de parámetros y conversaciones. |
| **SQL** (`ai_query`) | Procesamiento en lote sobre tablas, dashboards, analistas. |
| **Funciones de IA** (`ai_summarize`, `ai_classify`...) | Tareas comunes con la mínima fricción. |
| **API REST OpenAI-compatible** | Integración desde otros lenguajes o herramientas externas. |

### Recomendaciones
- **Controla costes**: revisa `usage` (tokens) y usa `max_tokens` para acotar respuestas. Los Foundation Model APIs se facturan por token.
- **`temperature` baja (0–0.2)** para tareas deterministas (clasificación, extracción); más alta para contenido creativo.
- **Versiona** tus dependencias (`databricks-sdk==x.y.z`) y los nombres de endpoint en producción.
- **No incrustes secretos**: deja que el SDK gestione la autenticación; fuera de Databricks usa Databricks Secrets o variables de entorno.
- Para producción a escala, evalúa **Provisioned Throughput** en lugar de pay-per-token.

### Recursos útiles
- Documentación de Mosaic AI Model Serving.
- Referencia de `ai_query` y funciones de IA en SQL.
- Repositorio del SDK de Python: `databricks/databricks-sdk-py`.